# ARGOS Multi-Tool ROM для Redmi Note 8T (ginkgo)
## Сборка LineageOS 21 + ARGOS overlay в Google Colab

**Runtime: GPU (или None)** — достаточно CPU.

**Время:** ~3-5 часов

**Диск:** 70GB SSD в Colab

In [ ]:
# Cell 1: Mount Google Drive для сохранения результата
from google.colab import drive
drive.mount('/content/drive')

# Создать папку для ROM
import os
os.makedirs('/content/drive/MyDrive/ARGOS_ROM', exist_ok=True)
print("Google Drive mounted. Результат будет сохранён в /content/drive/MyDrive/ARGOS_ROM/")

In [ ]:
# Cell 2: Установка зависимостей
!apt-get update -qq
!apt-get install -y -qq \
    git git-lfs repo bc bison build-essential ccache curl flex g++-multilib \
    gcc-multilib gperf imagemagick lib32ncurses-dev lib32readline-dev \
    lib32z1-dev libelf-dev liblz4-tool libsdl1.2-dev libssl-dev \
    libxml2 libxml2-utils lzop pngcrush rsync schedtool squashfs-tools \
    xsltproc zip zlib1g-dev python3 python3-pip openjdk-17-jdk gradle \
    &gt; /dev/null 2&gt;&1

# repo
!mkdir -p ~/bin
!curl -sL https://storage.googleapis.com/git-repo-downloads/repo > ~/bin/repo
!chmod a+x ~/bin/repo
import os; os.environ['PATH'] = os.path.expanduser('~/bin') + ':' + os.environ['PATH']

!git config --global user.email "argos@build.local"
!git config --global user.name "ARGOS Builder"

print("Dependencies installed!")

In [ ]:
# Cell 3: Инициализация LineageOS 21
import os
os.chdir('/content')
!mkdir -p lineageos && cd lineageos && repo init -u https://github.com/LineageOS/android.git -b lineage-21.0 --git-lfs --no-clone-bundle

In [ ]:
# Cell 4: Добавление device manifests для ginkgo
manifest = """<?xml version=\"1.0\" encoding=\"UTF-8\"?>
<manifest>
  <project name=\"LineageOS/android_device_xiaomi_ginkgo\" path=\"device/xiaomi/ginkgo\" remote=\"github\" revision=\"lineage-21\" />
  <project name=\"LineageOS/android_device_xiaomi_sm6150-common\" path=\"device/xiaomi/sm6150-common\" remote=\"github\" revision=\"lineage-21\" />
  <project name=\"LineageOS/android_kernel_xiaomi_ginkgo\" path=\"kernel/xiaomi/ginkgo\" remote=\"github\" revision=\"lineage-21\" />
  <project name=\"LineageOS/android_vendor_xiaomi_ginkgo\" path=\"vendor/xiaomi/ginkgo\" remote=\"github\" revision=\"lineage-21\" />
  <project name=\"LineageOS/android_hardware_xiaomi\" path=\"hardware/xiaomi\" remote=\"github\" revision=\"lineage-21\" />
</manifest>"""

with open('/content/lineageos/.repo/local_manifests/argos_ginkgo.xml', 'w') as f:
    f.write(manifest)

print("Device manifests added for ginkgo")

In [ ]:
# Cell 5: Sync LineageOS source (~45 мин)
import os
os.chdir('/content/lineageos')
!repo sync -c -j$(nproc --all) --force-sync --no-clone-bundle --optimized-fetch --prune

In [ ]:
# Cell 6: Применение ARGOS kernel patches
kernel_config = """
CONFIG_USB_SERIAL=y
CONFIG_USB_SERIAL_CH341=y
CONFIG_USB_SERIAL_FTDI_SIO=y
CONFIG_USB_SERIAL_CP210X=y
CONFIG_USB_SERIAL_PL2303=y
CONFIG_CAN=y
CONFIG_CAN_MCP251X=y
CONFIG_USB_ACM=y
CONFIG_USB_OTG=y
CONFIG_USB_OTG_WHITELIST=y
"""

config_path = '/content/lineageos/kernel/xiaomi/ginkgo/arch/arm64/configs/ginkgo_defconfig'
if os.path.exists(config_path):
    with open(config_path, 'a') as f:
        f.write(kernel_config)
    print(f"Kernel config patched: {config_path}")
else:
    # Try alternative path
    alt_paths = [
        '/content/lineageos/kernel/xiaomi/ginkgo/arch/arm64/configs/ginkgo_defconfig',
        '/content/lineageos/kernel/xiaomi/ginkgo/arch/arm64/configs/sm6150_defconfig'
    ]
    found = False
    for p in alt_paths:
        if os.path.exists(p):
            with open(p, 'a') as f:
                f.write(kernel_config)
            print(f"Kernel config patched: {p}")
            found = True
            break
    if not found:
        print("WARNING: Kernel defconfig not found. Search manually.")
        !find /content/lineageos/kernel/xiaomi/ginkgo -name "*defconfig*" | head -5

In [ ]:
# Cell 7: Сборка ROM (~2-3 часа)
import os
os.chdir('/content/lineageos')

# Setup ccache
os.environ['USE_CCACHE'] = '1'
os.environ['CCACHE_EXEC'] = '/usr/bin/ccache'
os.environ['CCACHE_DIR'] = '/content/ccache'
!ccache -M 10G

# Build
!source build/envsetup.sh && lunch lineage_ginkgo-userdebug && mka bacon -j$(nproc --all)

In [ ]:
# Cell 8: Копирование результата на Google Drive
import shutil
import glob

# Find built ROM
rom_files = glob.glob('/content/lineageos/out/target/product/ginkgo/*.zip')
boot_files = glob.glob('/content/lineageos/out/target/product/ginkgo/boot.img')

print(f"ROM files found: {len(rom_files)}")
print(f"Boot images found: {len(boot_files)}")

for f in rom_files + boot_files:
    dest = f"/content/drive/MyDrive/ARGOS_ROM/{os.path.basename(f)}"
    shutil.copy2(f, dest)
    print(f"Copied: {f} -> {dest}")

# Also copy to direct download
if rom_files:
    from google.colab import files
    files.download(rom_files[0]) if len(rom_files) == 1 else print("Multiple ROMs found. Check Drive.")
else:
    print("No ROM zip found. Build may have failed.")

In [ ]:
# Cell 9: Статус сборки (проверить если не знаешь что произошло)
import glob
import os

out_dir = '/content/lineageos/out/target/product/ginkgo'
if os.path.exists(out_dir):
    files = os.listdir(out_dir)
    print(f"Files in {out_dir}:")
    for f in sorted(files):
        fpath = os.path.join(out_dir, f)
        size = os.path.getsize(fpath) / (1024*1024*1024)
        print(f"  {f}: {size:.2f} GB")
else:
    print("Output directory not found. Build failed or not started.")